# Landing → Bronze

Lê os CSVs brutos da camada Landing (MinIO) e os grava como tabelas Delta Lake na camada Bronze.

**Sem transformações** — o Bronze preserva os dados no estado original, apenas no formato Delta.

> `spark`, `sc` e `minio` já estão disponíveis via startup script.

## 1. Configuração das tabelas

In [13]:
TABELAS = [
    "1_dim_partido",
    "2_dim_cargo",
    "3_dim_grau_instrucao",
    "4_dim_ocupacao",
    "5_dim_municipio",
    "6_fato_candidato",
    "7_dim_tipo_bem",
    "8_fato_bem",
    "9_agg_bens",
    "10_dim_coligacao",
]

LANDING_PATH = "s3a://landing/tse"
BRONZE_PATH  = "s3a://bronze/tse"

print(f"Landing : {LANDING_PATH}")
print(f"Bronze  : {BRONZE_PATH}")
print(f"Tabelas : {len(TABELAS)}")

Landing : s3a://landing/tse
Bronze  : s3a://bronze/tse
Tabelas : 10


## 2. Função de ingestão

In [14]:
def landing_to_bronze(tabela: str) -> int:
    origem  = f"{LANDING_PATH}/{tabela}.csv"
    destino = f"{BRONZE_PATH}/{tabela}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("encoding", "UTF-8")
        .csv(origem)
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(destino)
    )

    total = df.count()
    print(f"  [{tabela}] {total:,} registros gravados em {destino}")
    return total

## 3. Ingestão — tabelas dimensão

In [15]:
dimensoes = [t for t in TABELAS if "dim" in t or "agg" in t]
print(f"Processando {len(dimensoes)} tabelas de dimensão...\n")

for tabela in dimensoes:
    landing_to_bronze(tabela)

print("\nDimensões concluídas.")

Processando 8 tabelas de dimensão...



  [1_dim_partido] 78,422 registros gravados em s3a://bronze/tse/1_dim_partido
  [2_dim_cargo] 78,422 registros gravados em s3a://bronze/tse/2_dim_cargo
  [3_dim_grau_instrucao] 78,422 registros gravados em s3a://bronze/tse/3_dim_grau_instrucao
  [4_dim_ocupacao] 78,422 registros gravados em s3a://bronze/tse/4_dim_ocupacao
  [5_dim_municipio] 78,422 registros gravados em s3a://bronze/tse/5_dim_municipio
  [7_dim_tipo_bem] 157,803 registros gravados em s3a://bronze/tse/7_dim_tipo_bem
  [9_agg_bens] 157,803 registros gravados em s3a://bronze/tse/9_agg_bens
  [10_dim_coligacao] 78,422 registros gravados em s3a://bronze/tse/10_dim_coligacao

Dimensões concluídas.


## 4. Ingestão — tabelas fato

In [16]:
fatos = [t for t in TABELAS if "fato" in t]
print(f"Processando {len(fatos)} tabelas fato...\n")

for tabela in fatos:
    landing_to_bronze(tabela)

print("\nFatos concluídos.")

Processando 2 tabelas fato...

  [6_fato_candidato] 78,422 registros gravados em s3a://bronze/tse/6_fato_candidato
  [8_fato_bem] 160,835 registros gravados em s3a://bronze/tse/8_fato_bem

Fatos concluídos.


## 5. Validação da camada Bronze

In [17]:
print("=== Validação Bronze ===")
print(f"{'Tabela':<30} {'Registros':>10}")
print("-" * 42)

total_geral = 0
for tabela in TABELAS:
    destino = f"{BRONZE_PATH}/{tabela}"
    try:
        df = spark.read.format("delta").load(destino)
        n = df.count()
        total_geral += n
        print(f"{tabela:<30} {n:>10,}")
    except Exception as e:
        print(f"{tabela:<30} {'ERRO':>10} — {e}")

print("-" * 42)
print(f"{'TOTAL':<30} {total_geral:>10,}")

=== Validação Bronze ===
Tabela                          Registros
------------------------------------------


26/06/18 00:58:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

1_dim_partido                      78,422
2_dim_cargo                        78,422
3_dim_grau_instrucao               78,422
4_dim_ocupacao                     78,422
5_dim_municipio                    78,422
6_fato_candidato                   78,422
7_dim_tipo_bem                    157,803
8_fato_bem                        160,835
9_agg_bens                        157,803
10_dim_coligacao                   78,422
------------------------------------------
TOTAL                           1,025,395
